# `json2vec` Hello World

This notebook trains the smallest useful `json2vec` model: two numeric Iris measurements predict the Iris species. The point is not accuracy. The point is to show the complete loop from records, to schema, to training, to prediction and embeddings. This example is intentionally flat; the next tutorials show why arrays matter.

Start with the normal training dependencies plus the bundled Iris JSONL buffer. The examples remove notebook logging noise so the rendered docs stay focused on model behavior.


In [1]:
import lightning.pytorch as lit
import polars as pl
import torch
from loguru import logger
from rich.pretty import pprint

import json2vec as j2v

logger.remove()

Load a tiny balanced slice of Iris rows. The schema field names match the DataFrame columns, so `json2vec` can infer the request queries.


In [2]:
records = pl.read_ndjson("docs/data/iris.jsonl").head(36)

records.head()

sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
7.0,3.2,4.7,1.4,"""versicolor"""
6.3,3.3,6.0,2.5,"""virginica"""
4.9,3.0,1.4,0.2,"""setosa"""
6.4,3.2,4.5,1.5,"""versicolor"""


The schema declares exactly what the model should read. `Number` fields become numeric tensorfields, and the `Category` field is a supervised target because `target=True` hides it from the input and asks the model to decode it.

In [3]:
model = j2v.Model.from_schema(
    j2v.Number("sepal_length"),
    j2v.Number("petal_length"),
    j2v.Category("species", target=True, max_vocab_size=4, topk=[2]),
    d_model=16,
    n_layers=1,
    n_heads=4,
    batch_size=8,
    embed=True,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.PolarsDataModule(
    model=model,
    train=records,
    validate=records,
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    observation_buffer_size=32,
    sample_rate=1.0,
)

Train for one deliberately small epoch. The tutorials keep batch and epoch counts hardcoded so the example remains quick to run in documentation builds.

In [4]:
trainer = lit.Trainer(
    accelerator="cpu",
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    enable_checkpointing=False,
    limit_train_batches=1,
    limit_val_batches=1,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.


`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.


/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.
`Trainer.fit` stopped: `max_epochs=1` reached.


Prediction uses the same nested batch shape as training: each outer item is one observation, and each observation contains one record.

In [5]:
batch = records.to_dicts()[:3]

In [6]:
pprint(model.predict(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.2867260277271271,
│   │   │   │   0.374195396900177,
│   │   │   │   0.300782173871994,
│   │   │   │   -0.03721438720822334,
│   │   │   │   -0.5126165747642517,
│   │   │   │   0.28038445115089417,
│   │   │   │   -0.09533163160085678,
│   │   │   │   0.1500147432088852,
│   │   │   │   0.1745312660932541,
│   │   │   │   -0.41290679574012756,
│   │   │   │   0.02412782423198223,
│   │   │   │   -0.22400137782096863,
│   │   │   │   0.12148673087358475,
│   │   │   │   0.01725187711417675,
│   │   │   │   -0.09175621718168259,
│   │   │   │   0.19426698982715607
│   │   │   ],
│   │   │   [
│   │   │   │   -0.24348872900009155,
│   │   │   │   0.365494966506958,
│   │   │   │   0.3194009065628052,
│   │   │   │   -0.016082121059298515,
│   │   │   │   -0.5119074583053589,
│   │   │   │   0.28987109661102295,
│   │   │   │   -0.0840703472495079,
│   │   │   │   0.1339573860168457,
│   │   │   │   0.1951991766691208,
│   │   │   │   -0.41429606080055237,
│   │   │   │   0.006163083948194981,
│   │   │   │   -0.2528024911880493,
│   │   │   │   0.08373294770717621,
│   │   │   │   0.02144281566143036,
│   │   │   │   -0.114486463367939,
│   │   │   │   0.19873347878456116
│   │   │   ],
│   │   │   [
│   │   │   │   -0.2938827872276306,
│   │   │   │   0.3728201389312744,
│   │   │   │   0.31573137640953064,
│   │   │   │   -0.04200320318341255,
│   │   │   │   -0.49501436948776245,
│   │   │   │   0.2835691273212433,
│   │   │   │   -0.08518321067094803,
│   │   │   │   0.15233926475048065,
│   │   │   │   0.18803760409355164,
│   │   │   │   -0.4116807281970978,
│   │   │   │   0.018137002363801003,
│   │   │   │   -0.2264559417963028,
│   │   │   │   0.08313717693090439,
│   │   │   │   0.003880710806697607,
│   │   │   │   -0.09689367562532425,
│   │   │   │   0.21096865832805634
│   │   │   ]
│   │   ]
│   },
│   'record/species': {
│   │   'state': {
│   │   │   'valued': [0.405048131942749, 0.4035177230834961, 0.4039897620677948],
│   │   │   'null': [0.04948655888438225, 0.04849779233336449, 0.048860128968954086],
│   │   │   'padded': [0.1299126297235489, 0.12860162556171417, 0.12903091311454773],
│   │   │   'masked': [0.26260027289390564, 0.2643074095249176, 0.26519930362701416],
│   │   │   'other': [0.15295252203941345, 0.15507543087005615, 0.1529197245836258]
│   │   },
│   │   'content': {
│   │   │   'value': ['virginica', 'virginica', 'virginica'],
│   │   │   'probability': [0.782761812210083, 0.7851017713546753, 0.7849588394165039],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.782761812210083},
│   │   │   │   │   {'label': 'setosa', 'probability': 0.14326153695583344}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.7851017713546753},
│   │   │   │   │   {'label': 'setosa', 'probability': 0.1388750821352005}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'virginica', 'probability': 0.7849588394165039},
│   │   │   │   │   {'label': 'setosa', 'probability': 0.14074747264385223}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

Embeddings are opt-in. Passing `embed=True` when constructing the model includes a root `record` vector in `model.predict(...)` for each input observation without changing the schema fields themselves.


The Rich display is the quickest way to verify what was built: array nodes, tensorfield nodes, targets, embeddings, and inferred queries all appear in the same tree.

In [7]:
model

Model [model] batch_size=8 d_model=16 parameters=18,153 arrays=1 fields=3 targets=1 embeds=1
`-- record [root] embed attention=mha n_layers=1 n_heads=4 n_linear=1
    |-- sepal_length [number] active query=[*].sepal_length
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    |-- petal_length [number] active query=[*].petal_length
    |    pooling=query weight=1 p_mask=0 p_prune=0 n_heads=4 n_linear=1
    |    jitter=0 n_bands=8 offset=4 objective=mae
    `-- species [category] active target query=[*].species
         pooling=query weight=1 p_mask=0 p_prune=1 n_heads=4 n_linear=1
         max_vocab_size=4 p_unavailable=0.01 topk=[2]